In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DateType, TimestampType
import pyspark.sql.functions as F

## Brands

In [0]:
catalog_name = "ecommerce"
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show()

In [0]:
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))
df_silver.show(10)

In [0]:
df_silver=df_silver.withColumn("brand_code",F.regexp_replace(F.col("brand_code"),r'[^A-Za-z0-9]',''))
df_silver.show(10)

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
anomalies = {
    'GROCERY':'GRCY',
    'BOOKS':'BKS',
    'TOYS':'TYS'
}

df_silver=df_silver.replace(anomalies, subset='category_code')
df_silver.select("category_code").distinct().show()

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

## Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupby("category_code").count().filter(F.col("count")>1)
display(df_duplicates)

In [0]:
df_silver=df_bronze.dropDuplicates(['category_code'])
df_silver=df_silver.withColumn("category_code", F.upper(F.col('category_code')))
display(df_silver)

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_category")

## Customers

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customers")
df_bronze.show(10)

In [0]:
df_bronze.filter(F.col("customer_id").isNull()).show()

In [0]:
# drop the null values from customer ID column
df_silver=df_bronze.dropna(subset=["customer_id"])

# Get row count
row_count=df_silver.count()

# print the result
print(f"Row Count : {row_count}")
df_silver.display()

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

## Products

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_product")
df_bronze.show(10)

In [0]:
# get the row and column count
row_count,column_count=df_bronze.count(),len(df_bronze.columns)

# print the result
print(f"Row Count : {row_count}")
print(f"Column Count : {column_count}")

In [0]:
df_bronze.select("weight_grams").show(5, truncate=False)

In [0]:
df_silver=df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"),"g","").cast(IntegerType())
    )

df_silver.select("weight_grams").show(5,truncate=False)

In [0]:
df_silver=df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType())
)

df_silver.select("length_cm").show(10)

In [0]:
df_silver=df_silver.withColumns(
    {
        "category_code": F.upper(F.col("category_code")),
        "brand_code":F.upper(F.col("brand_code"))

    }
)

df_silver.show(10)

In [0]:
df_silver.select("material").distinct().show()

In [0]:
df_silver=df_silver.withColumn(
    "material",
    F.when(F.col("material") == "Coton","Cotton")
    .when(F.col("material")=="Alumium","Aluminium")
    .when(F.col("material")== "Ruber","Rubber")
    .otherwise(F.col("material"))
)

df_silver.select("material").distinct().show()

In [0]:
# rating count cannot be negative
# Therfore, convert the negative rating count to positive
df_silver = df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(),F.abs(F.col("rating_count")))
    .otherwise(F.lit(0))  # if null fill it with zero
) 
df_silver.select("rating_count").distinct().show()

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_product")

## Calender/date

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_date")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
# convert date column into date type
# import the function
from pyspark.sql.functions import to_date

# convert string column in to date type
df_silver=df_bronze.withColumn(
    "date",
    to_date(F.col("date"),"dd-MM-yyyy")
)

df_silver.printSchema()
df_silver.show()

In [0]:
df_duplicate=df_silver.groupBy("date").count().filter("count>1")
display(df_duplicate)

In [0]:
df_silver=df_silver.dropDuplicates(['date'])

# sanity check
df_duplicate=df_silver.groupBy("date").count().filter("count>1")
display(df_duplicate)

In [0]:
# Capitalize first letter of each word in day_name
df_silver=df_silver.withColumn("day_name",F.initcap(F.col("day_name")))
df_silver.show(5)

In [0]:
# convert negative week on the year to positive
df_silver=df_silver.withColumn("week_of_year",F.abs(F.col("week_of_year")))
df_silver.show(5)

In [0]:
# enhance week on the year pattern
df_silver=df_silver.withColumn(
    "week_of_year",
    F.concat(F.lit("Week"),F.col("week_of_year"),F.lit("-"),F.col("year"))
)

df_silver=df_silver.withColumn(
    "quarter",
    F.concat(F.lit("Q"),F.col('quarter'),F.lit("-"),F.col("year"))
)

In [0]:
df_silver=df_silver.withColumnRenamed("week_of_year","week")

In [0]:
# Sanity check
df_silver.show(10)

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeschema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_date")